In [ ]:
import pandas as pd
import requests
from arcgis.gis import GIS
from arcgis.features import FeatureSet, Feature
from arcgis.geometry import Point
import json
import os
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("WAQI_TOKEN")
API_KEY = os.getenv("ARCGIS_API_KEY")

# Connect to ArcGIS
gis = GIS(api_key=API_KEY)
print("Connected to ArcGIS:", gis)

Connected to ArcGIS: GIS @ https://www.arcgis.com version:[2026, 1]


In [4]:
import pandas as pd
import requests

TOKEN = "bb897fa6d8350fc2feb318abfc367eda6056adc7"  # your WAQI token

cities = ["houston", "arlington", "dallas", "austin", "san antonio", "el paso", 
          "fort worth", "lubbock", "midland", "beaumont", "corpus christi"]

stations = []
for city in cities:
    response = requests.get(f"https://api.waqi.info/search/?token={TOKEN}&keyword={city} texas")
    data = response.json()
    if data["status"] == "ok":
        for station in data["data"]:
            name = station["station"]["name"]
            aqi = station["aqi"]
            geo = station["station"]["geo"]
            if "Texas" in name and aqi != "-" and aqi != None:
                stations.append({
                    "name": name,
                    "aqi": int(aqi),
                    "lat": geo[0],
                    "lon": geo[1],
                    "url": station["station"]["url"]
                })

df = pd.DataFrame(stations).drop_duplicates(subset="name")

def categorize_aqi(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Moderate"
    elif aqi <= 150: return "Unhealthy for Sensitive Groups"
    else: return "Unhealthy"

df["category"] = df["aqi"].apply(categorize_aqi)
print(f"Stations loaded: {len(df)}")

Stations loaded: 32


In [5]:
m3 = gis.map("Texas, USA")
m3.zoom = 6

category_styles = {
    "Good":      (df[df["aqi"] <= 50],                        [0, 196, 0, 255]),
    "Moderate":  (df[(df["aqi"] > 50) & (df["aqi"] <= 100)], [255, 255, 0, 255]),
    "Unhealthy": (df[df["aqi"] > 100],                        [255, 0, 0, 255]),
}

for label, (subset, color) in category_styles.items():
    if subset.empty:
        continue
    features = []
    for _, row in subset.iterrows():
        features.append({
            "geometry": {
                "x": row["lon"],
                "y": row["lat"],
                "spatialReference": {"wkid": 4326}
            },
            "attributes": {
                "name": row["name"],
                "aqi": row["aqi"],
                "category": row["category"]
            }
        })
    fset = FeatureSet(features, geometry_type="esriGeometryPoint",
                      spatial_reference={"wkid": 4326})
    m3.content.add(fset, drawing_info={
        "renderer": {
            "type": "simple",
            "symbol": {
                "type": "esriSMS",
                "style": "esriSMSCircle",
                "color": color,
                "size": 14,
                "outline": {"color": [0, 0, 0, 255], "width": 1}
            },
            "label": label
        }
    })

m3

Map()

In [6]:
m3.export_to_html("texas_air_quality_map.html")
print("Exported!")

Exported!
